# 🌌 Galaxy Image Generation Using Diffusion Models

Date last modified: 1/7/24

Hi! My name is Jacob Nowack and I am the creator of this notebook. I am an undergraduate at Southern Oregon University studying Computer Science, and was lucky enough to get the opportunity to work with the great people at the UCLA Astrophysics Data Lab!

There were many very important contributors to this project including Andrew Lizarraga, Eric Hanchen Jiang, Yun Qi Li, Ying Nian Wu, Bernie Boscoe, and Tuan Do.

We have written a research paper diving into the specifics of how we designed our DDPM model and evaluated the results, along with the potential impact this project has
on our understanding of galaxy evolution. This paper has been submitted to, and accepted by Neural Information Processing Systems. Check it out!

Link to research paper: https://arxiv.org/pdf/2411.18440

Be sure to check out our other notebooks "Training" and "Evaluate" to dive deeper into the process of how this AI model was created, trained, and evaluated. The notebooks are available on our website!

UCLA Astrophysics Data Lab: https://datalab.astro.ucla.edu/


## Initial Setup
*This section imports all the necessary libraries and tools we'll need to generate galaxies. It includes extensive verification to make sure all dependencies have been properly downloaded, along with the trained model.*

**IMPORTANT NOTE**: *In Colab please click Runtime -> Change runtime type -> T4 GPU or higher otherwise notebook execution will be very slow*

In [1]:
# Import required libraries
import os
import sys
import shutil
import torch
import numpy as np
from pathlib import Path
import subprocess
import warnings
from IPython.display import clear_output, HTML, display
import time
from numpy._core.multiarray import _reconstruct

warnings.filterwarnings('ignore')

def create_status_html(message, status="info"):
    """Create HTML for status messages"""
    icons = {
        "info": "ℹ️",
        "success": "✅",
        "warning": "⚠️",
        "error": "❌",
        "loading": "⏳"
    }
    bg_colors = {
        'info': '#2196F3',
        'success': '#4CAF50',
        'warning': '#FF9800',
        'error': '#F44336',
        'loading': '#9C27B0'
    }

    icon = icons.get(status, "ℹ️")
    bg_color = bg_colors.get(status, '#2196F3')

    html = (
        f'<div style="padding: 10px; margin: 5px 0; border-radius: 5px; '
        f'background-color: {bg_color}; color: white; '
        f'font-family: monospace; font-weight: 500;">'
        f'{icon} {message}'
        f'</div>'
    )
    return html

def create_progress_html(progress, text=""):
    """Create HTML for progress bar"""
    bar_width = 50
    progress = min(1.0, progress)
    filled_width = int(bar_width * progress)
    bar = '█' * filled_width + '░' * (bar_width - filled_width)
    percentage = progress * 100

    html = (
        f'<div style="width: 100%; padding: 10px; margin: 5px 0; '
        f'border-radius: 5px; background-color: #1976D2; color: white; '
        f'font-family: monospace; font-weight: 500;">'
        f'{text}<br>'
        f'[{bar}] {percentage:.1f}%'
        f'</div>'
    )
    return html

def display_status(message, status="info"):
    """Display a status message with an icon"""
    display(HTML(create_status_html(message, status)))

def display_progress_bar(progress, text=""):
    """Display a custom progress bar"""
    display(HTML(create_progress_html(progress, text)))

def run_command(command, timeout=None):
    """Execute a command and return its output"""
    try:
        result = subprocess.run(command, shell=True, check=True,
                              capture_output=True, text=True,
                              timeout=timeout)
        return result.stdout
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds: {command}")
        return None
    except Exception as e:
        print(f"Error executing command: {e}")
        return None

def run_pip_install(package):
    """Run pip install with progress indication"""
    try:
        # For torch, show download size info
        if 'torch' in package:
            display_status("PyTorch is a large package (~2GB). This may take 3-5 minutes...", "info")
            result = subprocess.run(f"pip install -q {package}",
                                 shell=True, check=True,
                                 capture_output=True, text=True,
                                 timeout=300)  # 5 minute timeout
        else:
            result = subprocess.run(f"pip install -q {package}",
                                 shell=True, check=True,
                                 capture_output=True, text=True,
                                 timeout=180)  # 3 minute timeout
        return True
    except subprocess.TimeoutExpired:
        display_status(f"Installation of {package} timed out. You may need to restart the runtime.", "error")
        return False
    except Exception as e:
        display_status(f"Error installing {package}: {str(e)}", "error")
        return False

def setup_environment():
    """Set up the basic environment and install core dependencies"""
    packages = [
        ("gdown", "Downloading tool"),
        ("torch torchvision", "PyTorch"),
        ("pandas", "Pandas"),
        ("tqdm", "Progress bars")
    ]

    display_status("Setting up environment...", "loading")

    for i, (package, desc) in enumerate(packages, 1):
        clear_output(wait=True)
        display_status("Setting up environment...", "loading")
        display_progress_bar(i/len(packages), f"Step {i}/{len(packages)}: Installing {desc}")

        if "torch" in package:
            display_status("Installing PyTorch (this is a large package, may take 3-5 minutes)...", "info")

        try:
            success = run_pip_install(package)
            if success:
                display_status(f"✓ {desc} installed", "success")
            else:
                display_status(f"Failed to install {desc}", "error")
                raise Exception(f"Failed to install {desc}")
        except Exception as e:
            display_status(f"Failed to install {desc}: {e}", "error")
            raise

def download_and_extract_tools():
    """Download and extract the Gen_Modeling_Tools"""
    display_status("Downloading tools...", "loading")

    tools_file_id = "1L-ot2DOb8NZmM65lOw7wZKUwhK3VTnVT"
    result = run_command(f"gdown {tools_file_id} -O Gen_Modeling_Tools.zip", timeout=180)

    if not os.path.exists("Gen_Modeling_Tools.zip"):
        display_status("Download failed", "error")
        raise FileNotFoundError("Failed to download tools")

    display_status("Extracting files...", "loading")
    run_command("unzip -o Gen_Modeling_Tools.zip -d /content/", timeout=60)

    tools_path = find_tools_directory()
    if not tools_path:
        display_status("Could not find valid tools directory", "error")
        raise FileNotFoundError("Invalid directory structure")

    return tools_path

def find_tools_directory():
    """Find the correct tools directory path"""
    possible_paths = [
        "/content/Gen_Modeling_Tools",
        "/content/Gen_Modeling_Tools/Gen_Modeling_Tools"
    ]

    for path in possible_paths:
        if os.path.exists(path) and all(
            os.path.exists(os.path.join(path, f))
            for f in ["data_manage.py", "modules.py", "requirements.txt"]
        ):
            return path
    return None

def install_requirements(tools_path):
    """Install requirements with better progress tracking"""
    display_status("Installing requirements...", "loading")

    requirements_path = os.path.join(tools_path, "requirements.txt")
    with open(requirements_path, "r") as f:
        requirements = [line.strip() for line in f
                      if line.strip() and not line.startswith('#')]

    display_status("Note: Some packages may take several minutes to install", "info")
    skipped_packages = []

    for i, req in enumerate(requirements, 1):
        # Skip tqdm and torch packages as they're already installed
        if any(pkg in req.lower() for pkg in ['tqdm', 'torch']):
            skipped_packages.append(req)
            continue

        clear_output(wait=True)
        display_status("Installing requirements...", "loading")
        display_progress_bar(i/len(requirements), f"Package {i}/{len(requirements)}: Installing {req}")

        try:
            success = run_pip_install(req)
            if not success:
                display_status(f"Failed to install {req}. Trying alternative method...", "warning")
                # Try alternative installation method
                alt_success = run_command(f"python -m pip install --no-cache-dir {req}")
                if not alt_success:
                    display_status(f"All installation attempts for {req} failed", "error")
        except Exception as e:
            display_status(f"Error installing {req}: {str(e)}", "error")

    if skipped_packages:
        display_status(f"Skipped pre-installed packages: {', '.join(skipped_packages)}", "info")
    display_status("Requirements installation completed", "success")

def setup_python_path(tools_path):
    """Set up Python path"""
    display_status("Setting up Python path...", "loading")

    if os.path.exists(tools_path):
        # Remove any existing paths that might conflict
        sys.path = [p for p in sys.path if 'Gen_Modeling_Tools' not in p]
        # Add the new path at the beginning
        sys.path.insert(0, tools_path)
        # Verify the module files in the correct location
        module_files = ['modules.py', 'data_manage.py', 'utils.py']
        for module in module_files:
            module_path = os.path.join(tools_path, module)
            if not os.path.exists(module_path):
                raise FileNotFoundError(f"Cannot find {module} at {module_path}")
            display_status(f"Found {module}", "success")

        # Force Python to look for modules again
        import importlib
        importlib.invalidate_caches()
        display_status("Python path configured", "success")
    else:
        display_status("Tools directory not found", "error")
        raise FileNotFoundError("Missing tools directory")

def download_model():
    """Download the model with progress indicator"""
    display_status("Downloading model...", "loading")
    model_file_id = "1hEFRWTpQXEjkhCY4VD1IRFCiJPTARC69"
    result = run_command(f"gdown {model_file_id} -O best_model.pth", timeout=300)

    if os.path.exists("best_model.pth"):
        display_status("Model downloaded successfully", "success")
        return True
    else:
        display_status("Failed to download model", "error")
        return False

def load_model():
    """Load the model"""
    display_status("Loading model...", "loading")

    # torch.serialization.add_safe_globals(['_reconstruct'])

    try:
        import modules
        display_status(f"Found modules at: {modules.__file__}", "info")
        from modules import UNet_conditional_conv, EMA

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        display_status(f"Using device: {device}", "info")

        model = UNet_conditional_conv(
            c_in=5, c_out=5, time_dim=256, y_dim=1
        ).to(device)


        torch.serialization.clear_safe_globals()

        torch.serialization.add_safe_globals([
            (_reconstruct, "numpy.core.multiarray._reconstruct"),
            np.ndarray,
        ])

        checkpoint = torch.load(
            "best_model.pth",
            map_location=device,
            weights_only=False
        )

        model.load_state_dict(checkpoint['model_state_dict'])

        ema = EMA(model, beta=0.995)
        if 'ema_state_dict' in checkpoint:
            ema.load_state_dict(checkpoint['ema_state_dict'])

        display_status("Model loaded successfully", "success")
        return model, ema

    except Exception as e:
        display_status(f"Error loading model: {str(e)}", "error")
        raise

def main():
    """Main setup function"""
    try:
        steps = [
            ("Environment setup", setup_environment),
            ("Tool download & extraction", download_and_extract_tools),
            ("Requirements installation", lambda: install_requirements(find_tools_directory())),
            ("Model download", download_model),
            ("Python path setup", lambda: setup_python_path(find_tools_directory())),
            ("Model loading", load_model)
        ]

        results = []
        total_steps = len(steps)

        display_status("Starting setup...", "info")

        for i, (desc, func) in enumerate(steps, 1):
            clear_output(wait=True)
            display_status("Overall Progress", "info")
            display_progress_bar((i-1)/total_steps, f"Completed {i-1}/{total_steps} steps")
            display_status(f"Current step: {desc}", "loading")

            result = func()
            if result:
                results.append(result)

            time.sleep(0.5)  # Short pause to show completion

        clear_output(wait=True)
        display_status("Overall Progress", "success")
        display_progress_bar(1.0, f"Completed {total_steps}/{total_steps} steps")
        display_status("✨ Setup completed successfully! ✨", "success")
        return results[-1] if results else None

    except Exception as e:
        display_status(f"Setup failed: {str(e)}", "error")
        raise

if __name__ == "__main__":
    clear_output(wait=True)  # Clear any previous output
    model, ema = main()

In [ ]:
# Additional required imports
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from tqdm import tqdm
import logging
import random

# Set up matplotlib for inline display
%matplotlib inline

# Suppress specific PyTorch warning
warnings.filterwarnings('ignore', category=FutureWarning, message='You are using `torch.load`.*')


## Essential Settings
*These are the most commonly adjusted parameters. More detailed settings can be found in each section, with a complete reference guide at the end of the notebook.*


In [ ]:
# Configuration dictionary
config = {
    # Generation Settings
    'num_images': 10,          # How many galaxies to generate
    'use_fixed_seed': False,   # Set True to use specific seed value
    'seed': None,              # Set to a number to use that specific seed
    'random_redshifts': True,  # True for random redshifts, False for linear spacing

    # Display Settings
    'use_cmap': True,          # True for heatmap, False for grayscale
    'cmap': 'inferno',         # Good options: 'magma', 'inferno', 'viridis', 'hot'

    # Quality Control Settings
    'max_std': 0.25,           # Noise filtering threshold
    'skip_filtering': False,   # Set True to disable noise filtering

    # Model Path
    'model_dir': '.',
    'model_name': 'best_model.pth'
}

In [ ]:
# Verify model path
model_path = os.path.join(config['model_dir'], config['model_name'])
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model checkpoint not found at: {model_path}")
print(f"Found model checkpoint at: {model_path}")

# Device info - use PyTorch's device selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set up seed
if config['use_fixed_seed'] and config['seed'] is not None:
    seed_to_use = config['seed']
    print(f"Using fixed seed: {seed_to_use}")
else:
    seed_to_use = random.randint(0, 1000000)
    print(f"Using random seed: {seed_to_use}")

# Apply the seed to both torch and numpy
torch.manual_seed(seed_to_use)
np.random.seed(seed_to_use)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed_to_use)

Found model checkpoint at: ./best_model.pth
Using device: cuda
Using random seed: 268021


## 🎲 Random Seeds and Image Generation

### What Does This Do?
1. **Galaxy Generation**: Controls randomness pattern:
  - New random galaxies each time, or
  - Consistent pattern that eventually repeats
2. **Redshift Distribution**:
  - Random values in range, or
  - Evenly spaced values (0.1, 0.2, 0.3...)
3. **Real Galaxies**: Always shows different comparison images

### Control Settings:
1. **Random vs. Fixed** (`use_fixed_seed`):
  - `False`: New random galaxies each time (default)
  - `True`: Use specific seed for repeatable pattern

2. **Seed Number** (`seed`):
  - Set to `None` for random generation
  - Set to any seed number you like to use that pattern

3. **Redshift Pattern** (`random_redshifts`):
  - `True`: Random redshift values
  - `False`: Evenly spaced redshifts

### 📝 Example Workflow:
1. **If you want truly random galaxies each time:**
```python
config['use_fixed_seed'] = False
config['seed'] = None
```

2. **If you like a pattern and want to use it again:**
```python
config['use_fixed_seed'] = True
config['seed'] = 14414  # Use any seed number you've saved
```

### 💡 Note:
* Fixed seeds create repeatable patterns, not exact copies
* Real galaxies are always randomly selected
* Write down seeds that generate interesting patterns!
* The code will print the random seed being used
* Fixed seeds are useful for:
  * Exploring consistent galaxy patterns
  * Sharing results with colleagues
  * Systematic analysis

## Image Processing
*This section contains functions that convert the images from the format the model uses (-1 to 1) to a format we can actually view (0 to 1). Think of it like developing a photo negative into a viewable picture.*


In [ ]:
def denormalize_images(images):
    """Converts images from model format (-1 to 1) to viewable format (0 to 1)"""
    return (images + 1) / 2

## The Denoising Diffusion Probabilistic Model
*This is the core of our galaxy generator. It's a deep learning model that generates images by learning and reversing a gradual noising process.*

**Process Overview:**
- Training: Model learns to reverse a sequence of noising steps applied to real galaxy images
- Generation: Starting from random noise, iteratively applies learned denoising to create new galaxies
- Each denoising step gradually reveals galaxy features and structure, from basic shape to fine details

**Adjustable Parameters:**
- `noise_steps`: Number of denoising iterations (default: 1000)
 - Higher values generally yield better quality at cost of generation time
- `beta_start` and `beta_end`: Control noise schedule/rate of denoising
 - Defines the variance schedule of the forward/reverse process
- `img_size`: Output image dimensions (64x64 pixels default)

In [ ]:
class Diffusion:
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02, img_size=64, device="cuda"):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.beta = self.prepare_noise_schedule().to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)
        self.img_size = img_size
        self.device = device

    def prepare_noise_schedule(self):
        """Prepares a cosine noise schedule for the diffusion process."""
        steps = torch.linspace(0, self.noise_steps, self.noise_steps)
        schedule = torch.cos((steps / self.noise_steps) * (torch.pi / 2))
        beta = self.beta_start + (self.beta_end - self.beta_start) * (1 - schedule)
        return beta

    def sample_images(self, model, n, labels):
        """Generates n galaxy images based on given redshift labels with a progress bar."""
        model.eval()
        with torch.no_grad():
            # Initialize starting noise for the images
            x = torch.randn((n, 5, self.img_size, self.img_size)).to(self.device)

            # Ensure labels are shaped correctly
            labels = labels.view(-1, 1) if labels.dim() == 1 else labels

            # Progress bar for the diffusion process
            for i in tqdm(reversed(range(1, self.noise_steps)), desc="Generating Images", total=self.noise_steps):
                t = (torch.ones(n) * i).long().to(self.device)
                predicted_noise = model(x, t, labels)

                # Calculate alphas and betas for the diffusion step
                alpha = self.alpha[t][:, None, None, None]
                alpha_hat = self.alpha_hat[t][:, None, None, None]
                beta = self.beta[t][:, None, None, None]

                # Sample noise at each step, except for the last step
                noise = torch.randn_like(x) if i > 1 else torch.zeros_like(x)

                # Update x with predicted noise and diffusion process adjustments
                x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / torch.sqrt(1 - alpha_hat)) * predicted_noise) + torch.sqrt(beta) * noise

        model.train()
        return x

## 🔍 Image Quality Control
*This section defines functions for filtering and displaying galaxy images across all 5 wavelength channels (g, r, i, z, y).*

**Quality Control Settings (set in config):**
- `skip_filtering`: Control noise filtering
 - `True`: Show all generated images
 - `False`: Filter out noisy images (default)
- `max_std`: Noise filtering threshold
 - Lower values = stricter filtering (e.g., 0.15)
 - Default = 0.25
 - Higher values = more permissive (e.g., 0.35)

**Visualization Options:**
1. **Channel View**
  - Shows all 5 channels (g, r, i, z, y) separately
  - Each row shows how galaxies appear at different wavelengths
  - Uses colormap settings from config

2. **RGB View**
  - Combines g, r, i channels into natural color image
  - Shows galaxies as they might appear to human eye

*Note: Quality control settings can be adjusted in the config section at the top of the notebook. Each function also accepts these parameters directly for custom filtering on specific plots.*

In [ ]:
def filter_and_replace_noisy_images(images, max_std=0.25, skip_filtering=False):
    """
    Removes overly noisy images and replaces them with good ones.

    Args:
        images: Image tensor to filter
        max_std: Maximum standard deviation threshold (default: from config or 0.25)
        skip_filtering: If True, returns images unfiltered (default: from config or False)
    """
    if skip_filtering:
        return images

    good_images = [img for img in images if img.std().item() < max_std]
    while len(good_images) < images.size(0):
        good_images.extend(good_images)
    good_images = good_images[:images.size(0)]
    return torch.stack(good_images)

def plot_all_channels(images, num_images, max_std=None, cmap=None, use_cmap=None, skip_filtering=None, redshifts=None):
    """
    Plots all 5 channels (g, r, i, z, y) with option for colormap or grayscale.

    Args:
        images: Image tensor of shape (N, 5, H, W)
        num_images: Number of images to display
        max_std: Maximum standard deviation for noise filtering (default: from config)
        cmap: Colormap to use (default: from config)
        use_cmap: If True, uses colormap (default: from config)
        skip_filtering: If True, disables noise filtering (default: from config)
        redshifts: Tensor of redshift values for each image
    """
    # Use config settings if not explicitly provided
    max_std = config['max_std'] if max_std is None else max_std
    cmap = config['cmap'] if cmap is None else cmap
    use_cmap = config['use_cmap'] if use_cmap is None else use_cmap
    skip_filtering = config['skip_filtering'] if skip_filtering is None else skip_filtering

    # Filter images if enabled
    images = filter_and_replace_noisy_images(images, max_std, skip_filtering)
    num_images = min(num_images, images.size(0))
    num_channels = images.size(1)

    # Create figure and grid
    fig = plt.figure(figsize=(num_images * 3, num_channels * 2.5))
    gs = fig.add_gridspec(num_channels, num_images, left=0.015, right=0.98,
                         top=0.95, bottom=0.05, wspace=0.1, hspace=0.1)

    axs = [[fig.add_subplot(gs[i, j]) for j in range(num_images)] for i in range(num_channels)]
    channel_names = ['g', 'r', 'i', 'z', 'y']

    # Plot each channel
    for channel_idx in range(num_channels):
        for img_idx in range(num_images):
            channel_data = images[img_idx, channel_idx].cpu().numpy()

            # Safe normalization
            channel_min = channel_data.min()
            channel_max = channel_data.max()
            if channel_max - channel_min > 1e-10:  # Check if range is non-zero
                channel_data = (channel_data - channel_min) / (channel_max - channel_min)
            else:
                channel_data = np.full_like(channel_data, 0.5)

            axs[channel_idx][img_idx].imshow(channel_data,
                                           cmap=cmap if use_cmap else 'gray')
            axs[channel_idx][img_idx].axis('off')

            if channel_idx == 0:
                if redshifts is not None:
                    title = f"redshift = {redshifts[img_idx].item():.2f}"
                else:
                    title = f"Image {img_idx + 1}"
                axs[channel_idx][img_idx].set_title(title)

            if img_idx == 0:
                fig.text(0.01, 0.85 - (channel_idx * 0.195), channel_names[channel_idx],
                        ha='left', va='center', fontsize=14, fontweight='bold')
    plt.show()

def plot_rgb_channels(images, num_images, max_std=None, skip_filtering=None, redshifts=None):
    """
    Plots images using g, r, i channels combined into RGB display.

    Args:
        images: Image tensor of shape (N, 5, H, W)
        num_images: Number of images to display
        max_std: Maximum standard deviation for filtering (default: from config)
        skip_filtering: If True, skips filtering (default: from config)
        redshifts: Tensor of redshift values for each image
    """
    # Use config settings if not explicitly provided
    max_std = config['max_std'] if max_std is None else max_std
    skip_filtering = config['skip_filtering'] if skip_filtering is None else skip_filtering

    # Filter images if enabled
    images = filter_and_replace_noisy_images(images, max_std, skip_filtering)
    num_images = min(num_images, images.size(0))

    # Calculate grid layout
    rows = (num_images // 6) + (1 if num_images % 6 else 0)
    fig, axs = plt.subplots(rows, 6, figsize=(16, rows * 4))
    axs = axs.flatten()

    for idx, ax in enumerate(axs):
        if idx >= num_images:
            ax.axis('off')
            continue

        # Combine g, r, i channels into RGB
        rgb_image = torch.stack([
            images[idx, 0],  # g -> Red channel
            images[idx, 1],  # r -> Green channel
            images[idx, 2]   # i -> Blue channel
        ], dim=0)

        # Normalize and convert to numpy
        rgb_image = rgb_image.permute(1, 2, 0).cpu().numpy()

        # Safe normalization for RGB image
        img_min = rgb_image.min()
        img_max = rgb_image.max()
        if img_max - img_min > 1e-10:  # Check if range is non-zero
            rgb_image = (rgb_image - img_min) / (img_max - img_min)
        else:
            rgb_image = np.full_like(rgb_image, 0.5)

        # Plot
        ax.imshow(rgb_image)
        ax.axis('off')

        if redshifts is not None:
            title = f"redshift = {redshifts[idx].item():.2f}"
        else:
            title = f"Image #{idx+1}"
        ax.set_title(title)

    plt.tight_layout()
    plt.show()

## 🛠️ Model Configuration
*This section initializes the model with optimal parameters for galaxy generation.*

**Model Parameters:**
- `image_size`: Size of generated images (64x64 pixels)
- `noise_steps`: Number of denoising steps (1000)
- `beta_start`, `beta_end`: Control noise reduction process
- `ema_decay`: Controls model stability

*Note: These parameters are pre-configured for optimal performance and rarely need adjustment.*

In [ ]:
class Args:
    def __init__(self):
        self.device = str(next(model.parameters()).device)  # Get device from existing model
        self.image_size = 64    # Size of generated images
        self.noise_steps = 1000  # Steps in diffusion process
        self.beta_start = 1e-4   # Starting noise level
        self.beta_end = 0.02     # Ending noise level

args = Args()
print(f"Using model on device: {args.device}")

Using model on device: cuda:0


## Galaxy Generation
*This section generates our galaxies across a specified redshift range.*

**Generation Settings:**
- Number of images: {config['num_images']} (set in config)
- Redshift range: Adjust `redshift_min` and `redshift_max` below to control galaxy properties
  - Lower redshifts: More evolved, nearby galaxies
  - Higher redshifts: Younger, more distant galaxies

*Note: Feel free to experiment with different redshift ranges to see how galaxy properties change!*

## Generate multiple galaxies per redshift and save in one HDF5 file

Set `redshift_values`, `images_per_redshift`, and `batch_size` below. The original `Diffusion.sample_images` method is reused; output is written incrementally to HDF5. **Running the cell again overwrites the destination file.**

In [ ]:
# Batch generation configuration: x galaxies for EVERY redshift in this vector.
import h5py
import numpy as np
import torch
from pathlib import Path

redshift_values = [1.0,1.1,1.2,1.3,1.4,1.5]  # Edit this vector
images_per_redshift = 100                  # x images for each r
batch_size = 16                            # Reduce if CUDA runs out of memory
output_file = "/content/generated_galaxies.hdf5"
spectrum_length = 7783                     # Match the downstream dataset

# Images are saved in the diffusion model's original numerical scale (typically [-1, 1]).
# They are NOT denormalized, clipped, or subjected to display-only quality filtering.

def generate_redshift_dataset(
    diffusion, ema_model, redshift_values, images_per_redshift,
    output_file, batch_size=16, spectrum_length=7783,
):
    redshift_values = np.asarray(redshift_values, dtype=np.float32).reshape(-1)
    if not len(redshift_values) or not np.all(np.isfinite(redshift_values)):
        raise ValueError("redshift_values must contain finite values")
    if images_per_redshift < 1 or batch_size < 1 or spectrum_length < 1:
        raise ValueError("images_per_redshift, batch_size, spectrum_length must be positive")

    total = len(redshift_values) * images_per_redshift
    output_file = str(output_file)
    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    height = width = diffusion.img_size
    model_device = next(ema_model.parameters()).device
    previous_mode = ema_model.training
    print(f"Generating {total} galaxies ({images_per_redshift} per redshift)")

    # 'w' creates/replaces the destination file. All batches append within it.
    with h5py.File(output_file, 'w') as f:
        image_ds = f.create_dataset(
            'image', shape=(total, 5, height, width), dtype='float32',
            chunks=(min(batch_size, total), 5, height, width),
            compression='gzip', compression_opts=4,
        )
        redshift_ds = f.create_dataset('specz_redshift', shape=(total,), dtype='float32')
        spectrum_ds = f.create_dataset(
            'spectrum', shape=(total, spectrum_length), dtype='float32',
            chunks=(min(batch_size, total), spectrum_length),
            compression='gzip', compression_opts=4, fillvalue=0.0,
        )
        # The spectrum dataset is intentionally left at its zero fill value.
        ids_ds = f.create_dataset('targetid', data=np.arange(total, dtype=np.int64))
        f.attrs['source'] = 'conditional_diffusion'
        f.attrs['image_layout'] = 'NCHW'
        f.attrs['channels'] = 'g,r,i,z,y'
        f.attrs['image_scale'] = 'raw_diffusion_output_typically_minus1_to_1'
        f.attrs['spectrum_type'] = 'all_zeros_missing_modality_placeholder'
        f.attrs['images_per_redshift'] = images_per_redshift
        f.attrs['redshift_values'] = redshift_values

        cursor = 0
        try:
            for r in tqdm(redshift_values, desc='Redshift values'):
                for offset in range(0, images_per_redshift, batch_size):
                    current_batch = min(batch_size, images_per_redshift - offset)
                    labels = torch.full(
                        (current_batch, 1), float(r),
                        dtype=torch.float32, device=model_device,
                    )
                    with torch.inference_mode():
                        images = diffusion.sample_images(ema_model, current_batch, labels)
                    images_np = images.detach().cpu().numpy().astype(np.float32)
                    if images_np.shape != (current_batch, 5, height, width):
                        raise ValueError(f'Unexpected image shape: {images_np.shape}')
                    image_ds[cursor:cursor + current_batch] = images_np
                    redshift_ds[cursor:cursor + current_batch] = r
                    cursor += current_batch
                    del images, images_np
                f.flush()
                print(f"Finished redshift {r:.5g}: {cursor}/{total} images")
        finally:
            ema_model.train(previous_mode)  # sample_images() sets model.train() on return

    print(f"Saved {total} samples to {output_file}")
    return output_file

# Initialize the diffusion sampler (the original generation cell was replaced).
# Requires the earlier cells defining Diffusion and args to have been run.
diffusion = Diffusion(
    noise_steps=args.noise_steps,
    beta_start=args.beta_start,
    beta_end=args.beta_end,
    img_size=args.image_size,
    device=args.device,
)

# The original sampling method still performs 999 reverse-diffusion steps per batch.
ema_model = ema.get_ema_model()
output_file = generate_redshift_dataset(
    diffusion=diffusion,
    ema_model=ema_model,
    redshift_values=redshift_values,
    images_per_redshift=images_per_redshift,
    output_file=output_file,
    batch_size=batch_size,
    spectrum_length=spectrum_length,
)


Generating 600 galaxies (100 per redshift)


Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 59.56it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 59.39it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 59.91it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.40it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.14it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.25it/s]

Redshift values:  17%|█▋        | 1/6 [01:48<09:00, 108.08s/it]

Finished redshift 1: 100/600 images



Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.13it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.22it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.18it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.32it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.45it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.39it/s]

Redshift values:  33%|███▎      | 2/6 [03:35<07:11, 107.82s/it]

Finished redshift 1.1: 200/600 images



Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.38it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.21it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.42it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 59.92it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 59.92it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 59.86it/s]

Redshift values:  50%|█████     | 3/6 [05:23<05:23, 107.91s/it]

Finished redshift 1.2: 300/600 images



Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.16it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.11it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.14it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.30it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.18it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 59.92it/s]

Redshift values:  67%|██████▋   | 4/6 [07:11<03:35, 107.99s/it]

Finished redshift 1.3: 400/600 images



Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 59.90it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.37it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.31it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 59.71it/s]

Generating Images: 100%|█████████▉| 999/1000 [00:16<00:00, 60.28it/s]

Redshift values:  67%|██████▋   | 4/6 [08:47<04:23, 131.97s/it]


KeyboardInterrupt: 

## Inspect saved data

In [ ]:
# # Verify the HDF5 structure and preview a small sample without loading the whole dataset.
# with h5py.File(output_file, 'r') as f:
#     for key in f:
#         print(key, f[key].shape, f[key].dtype)
#     count = min(6, len(f['image']))
#     preview_images = torch.from_numpy(f['image'][:count])
#     preview_redshifts = torch.from_numpy(f['specz_redshift'][:count])

# # Visualization only: convert from diffusion scale to [0, 1].
# preview_display = torch.clamp(denormalize_images(preview_images), 0.0, 1.0)
# plot_rgb_channels(
#     images=preview_display, num_images=count,
#     skip_filtering=True, redshifts=preview_redshifts,
# )
# plot_all_channels(
#     images=preview_display, num_images=count,
#     skip_filtering=True, redshifts=preview_redshifts,
# )


## Download dataset

In [ ]:
# Download the single HDF5 file from Google Colab.
from google.colab import files
files.download(output_file)

# Optional: persist it to Google Drive instead of downloading it:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy2(output_file, '/content/drive/MyDrive/generated_galaxies.hdf5')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📊 Visualize Real Images For Comparison

**IMPORTANT NOTE**: *This section will not run in Colab due to the absence of the testing dataset. However, if you wish to download the GalaxiesML dataset along with this notebook you can run them together on your local machine or server! For now though, just go ahead and skip this cell*



*This reproduces the same approach we used above, but instead of displaying generated images it extracts real galaxy images from our testing dataset and displays them in the exact same way for easy visual comparison. You can easily edit the redshift range for the real images by changing the values for redshift_min and redshift_max*

In [ ]:
# def load_real_images(hdf5_path, num_images, redshift_min=None, redshift_max=None, random_redshifts=True, seed_to_use=None):
#     """
#     Load real images from HDF5 file with optional redshift filtering.
#     """
#     # Generate a completely new random seed for this specific selection
#     selection_seed = random.randint(0, 1000000)
#     np.random.seed(selection_seed)


#     with h5py.File(hdf5_path, 'r') as f:
#         total_images = f['image'].shape[0]
#         all_redshifts = f['specz_redshift'][:]

#         # Filter by redshift range
#         valid_indices = np.where((all_redshifts >= redshift_min) &
#                                (all_redshifts <= redshift_max))[0]

#         if len(valid_indices) == 0:
#             raise ValueError(f"No images found in redshift range {redshift_min} to {redshift_max}")

#         # Select images
#         if random_redshifts:
#             print("Using random selection")
#             selected_indices = np.random.permutation(valid_indices)[:num_images]
#         else:
#             print("Using linear spacing with random selection at each redshift")
#             target_redshifts = np.linspace(redshift_min, redshift_max, num_images)
#             selected_indices = []

#             for target_z in target_redshifts:
#                 # Find all images near this redshift
#                 redshift_diffs = np.abs(all_redshifts[valid_indices] - target_z)
#                 # Get the indices of the 10 closest matches
#                 closest_matches = np.argsort(redshift_diffs)[:10]  # Take 10 closest matches
#                 # Randomly select one from these matches
#                 selected_idx = valid_indices[closest_matches[np.random.randint(0, len(closest_matches))]]
#                 selected_indices.append(selected_idx)

#             selected_indices = np.array(selected_indices)

#         # Sort indices for HDF5 access
#         sorted_idx = np.argsort(selected_indices)
#         sorted_selected_indices = selected_indices[sorted_idx]

#         # Load images and redshifts
#         selected_images = f['image'][sorted_selected_indices]
#         selected_redshifts = f['specz_redshift'][sorted_selected_indices]

#         # Restore original order
#         selected_images = selected_images[np.argsort(sorted_idx)]
#         selected_redshifts = selected_redshifts[np.argsort(sorted_idx)]

#         # Reset random seed to original value if one was provided
#         if seed_to_use is not None:
#             np.random.seed(seed_to_use)

#         return torch.tensor(selected_images), torch.tensor(selected_redshifts)

In [ ]:
# # Define the path to the testing HDF5 file
# testing_hdf5_path = '/shared/astrodata/5x64x64_testing_with_morphology.hdf5'

# # Define redshift range (similar to generated images)
# redshift_min, redshift_max = 0.1, .5

# print(f"\nLoading real galaxies in redshift range: {redshift_min} to {redshift_max}")

# try:
#     real_images_tensor, real_redshifts = load_real_images(
#         testing_hdf5_path,
#         config['num_images'],
#         redshift_min=redshift_min,
#         redshift_max=redshift_max,
#         random_redshifts=config['random_redshifts'],
#         seed_to_use=seed_to_use
#     )

#     # Normalize the real images for display
#     real_images_denorm = denormalize_images(real_images_tensor)
#     real_images_denorm = torch.clamp(real_images_denorm, 0.0, 1.0)

#     # Display RGB composite view for real images
#     print("\n🎨 RGB Composite View (g, r, i channels combined) - Real Images:")
#     plot_rgb_channels(
#         images=real_images_denorm,
#         num_images=config['num_images'],
#         max_std=config['max_std'],
#         skip_filtering=config['skip_filtering'],
#         redshifts=real_redshifts
#     )

#     # Display individual channel view for real images
#     print("\n📊 Individual Channel View (g, r, i, z, y separately) - Real Images:")
#     plot_all_channels(
#         images=real_images_denorm,
#         num_images=config['num_images'],
#         max_std=config['max_std'],
#         cmap=config['cmap'],
#         use_cmap=config['use_cmap'],
#         skip_filtering=config['skip_filtering'],
#         redshifts=real_redshifts
#     )

#     # Display statistics and settings for real images
#     print("\n📊 Image Statistics - Real Images:")
#     print(f"Mean intensity: {real_images_denorm.mean().item():.4f}")
#     print(f"Standard deviation: {real_images_denorm.std().item():.4f}")
#     print(f"Redshift range: {real_redshifts.min().item():.2f} to {real_redshifts.max().item():.2f}")

#     print("\n Current Settings for Real Images:")
#     print(f"  • Redshift Distribution: {'Random' if config['random_redshifts'] else 'Linear'}")
#     print(f"  • Redshift Range: {redshift_min} to {redshift_max}")
#     print("\nDisplay Settings:")
#     print(f"  • Mode: {'Heatmap' if config['use_cmap'] else 'Grayscale'}")
#     print(f"  • Colormap: {config['cmap'] if config['use_cmap'] else 'N/A'}")
#     print("\nQuality Control:")
#     print(f"  • Noise Filtering: {'Disabled' if config['skip_filtering'] else 'Enabled'}")
#     print(f"  • Filter Threshold: {config['max_std']} (lower = stricter)")

# except ValueError as e:
#     print(f"Error: {e}")
#     print("Please try a different redshift range.")

# Done
The generated dataset is stored in one HDF5 file with `image`, `specz_redshift`, `spectrum`, and `targetid` datasets. The optional real-galaxy comparison above requires the original external test dataset.
